In [2]:
import asyncio
import os
import re
from pathlib import Path
from playwright.async_api import async_playwright
import pymupdf
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
URL = "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/visa-denials.html"

async def expand_accordions(page):

    # Click all accordion headers
    headers = await page.query_selector_all(".tsg-rwd-accordion-header")

    print(f"Accordion headers found: {len(headers)}")

    for h in headers:
        try:
            await h.scroll_into_view_if_needed()
            await h.click()
            await asyncio.sleep(0.2)
        except:
            pass

    # Force open anything still collapsed (JS override)
    await page.evaluate("""
        document.querySelectorAll('.tsg-rwd-accordion-header').forEach(el => {
            el.setAttribute('aria-expanded', 'true');
        });

        document.querySelectorAll('.tsg-rwd-accordion-content').forEach(el => {
            el.style.display = 'block';
            el.style.maxHeight = 'none';
        });
    """)

    await asyncio.sleep(1)


async def scrape_one():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle")

        await asyncio.sleep(2)

        await expand_accordions(page)

        # Save PDF
        await page.pdf(
            path="visa_denials.pdf",
            format="A4",
            print_background=True
        )

        print("✅ PDF saved")

        await browser.close()

    # Extract text from PDF
    doc = pymupdf.open("visa_denials.pdf")

    text = ""
    for page in doc:
        text += page.get_text()

    print("Characters:", len(text))
    print("Pages:", len(doc))

    print("\n", "="*60)
    print(text[:3000])


await scrape_one()

/Users/arulnanda/Desktop/LLM/USVISA_RAG/venv/lib/python3.11/site-packages/pyee/base.py:194: RuntimeWarning: coroutine 'scrape_visa_denials' was never awaited
  funcs = list(self._events.get(event, OrderedDict()).values())


Accordion headers found: 13
✅ PDF saved
Characters: 16293
Pages: 10

Visa Denials
U.S. law generally requires visa applicants to be interviewed by
a consular officer at a U.S. Embassy or Consulate. After relevant
information is reviewed, the application is approved or denied,
based on standards established in U.S. law.
While the vast majority of visa applications are approved, U.S.
law sets out many standards under which a visa application may
be denied. An application may be denied because the consular
officer does not have all of the information required to
determine if the applicant is eligible to receive a visa, because
the applicant does not qualify for the visa category for which he
or she applied, or because the information reviewed indicates
the applicant falls within the scope of one of the inadmissibility
or ineligibility grounds of the law. An applicant’s current and/or
past actions, such as drug or criminal activities, as examples,
may make the applicant ineligible for a vi

Source URLs

In [7]:
visa_rag_sources = {
    "visa_types": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html"
    ],
    "ds160_application": [
        "https://ceac.state.gov/genniv/",
        "https://ceac.state.gov/genniv/Common/Instructions.aspx",
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application.html",
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application/ds-160-faqs.html#doclist"
    ],
    "photo_requirements": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/photos.html"
    ],
    "visa_fees": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/fees/fees-visa-services.html"
    ],
    "visa_process_ustraveldocs": [
        "https://www.ustraveldocs.com/in/en/step-1",
        "https://www.ustraveldocs.com/in/en/step-2",
        "https://www.ustraveldocs.com/in/en/step-3",
        "https://www.ustraveldocs.com/in/en/step-4",
        "https://www.ustraveldocs.com/in/en/step-5",
        "https://www.ustraveldocs.com/in/en/step-6"
    ],
    "administrative_processing": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/administrative-processing-information.html"
    ],
    "visa_denials": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/visa-denials.html"
    ],
    "visa_wait_times": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/wait-times.html"
    ],
    "us_embassy_india": [
        "https://www.usembassy.gov/india/"
    ]
}

State Gov website -> PDF

In [8]:
# ── Output folder ─────────────────────────────────────────────────────────────
PDF_DIR = Path("PDFs")
PDF_DIR.mkdir(exist_ok=True)

def url_to_filename(category: str, url: str, index: int) -> str:
    """Convert a URL + category into a clean PDF filename."""
    # Strip scheme and www
    clean = re.sub(r"https?://(www\.)?", "", url)
    # Keep only alphanumerics, hyphens, underscores
    clean = re.sub(r"[^\w\-]", "_", clean)
    # Collapse multiple underscores and trim
    clean = re.sub(r"_+", "_", clean).strip("_")
    # Truncate so filenames stay reasonable
    clean = clean[:80]
    return f"{category}__{index+1}__{clean}.pdf"


async def expand_accordions(page):
    """Expand all tsg-rwd-accordion-header sections (works for state.gov pages)."""
    headers = await page.query_selector_all(".tsg-rwd-accordion-header")
    if headers:
        print(f"    📂 Accordion headers found: {len(headers)}")
        for h in headers:
            try:
                await h.scroll_into_view_if_needed()
                await h.click()
                await asyncio.sleep(0.2)
            except:
                pass

    # JS override — force-open any remaining collapsed content
    await page.evaluate("""
        document.querySelectorAll('.tsg-rwd-accordion-header').forEach(el => {
            el.setAttribute('aria-expanded', 'true');
        });
        document.querySelectorAll('.tsg-rwd-accordion-content').forEach(el => {
            el.style.display = 'block';
            el.style.maxHeight = 'none';
        });
    """)
    await asyncio.sleep(1)


async def scrape_page(page, url: str, pdf_path: Path):
    """Navigate to a URL, expand accordions, save as PDF."""
    try:
        await page.goto(url, wait_until="networkidle", timeout=30000)
    except Exception as e:
        print(f"    ⚠️  networkidle timed out ({e}), continuing anyway...")

    await asyncio.sleep(2)
    await expand_accordions(page)

    await page.pdf(
        path=str(pdf_path),
        format="A4",
        print_background=True,
        margin={"top": "1cm", "bottom": "1cm", "left": "1cm", "right": "1cm"},
    )


async def scrape_all():
    results = []  # (category, url, pdf_path, pages, chars) or error info

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        for category, urls in visa_rag_sources.items():
            print(f"\n{'='*60}")
            print(f"📁 Category: {category}  ({len(urls)} URL{'s' if len(urls)>1 else ''})")

            for idx, url in enumerate(urls):
                filename = url_to_filename(category, url, idx)
                pdf_path = PDF_DIR / filename
                print(f"\n  [{idx+1}/{len(urls)}] {url}")
                print(f"    💾 → {pdf_path}")

                # Fresh page per URL (avoids state bleed between sites)
                page = await browser.new_page(
                    viewport={"width": 1280, "height": 900}
                )

                try:
                    await scrape_page(page, url, pdf_path)

                    # Quick stats via PyMuPDF
                    doc = pymupdf.open(str(pdf_path))
                    chars = sum(len(pg.get_text()) for pg in doc)
                    pages = len(doc)
                    doc.close()

                    print(f"    ✅ {pages} pages | {chars:,} chars")
                    results.append((category, url, pdf_path, pages, chars, None))

                except Exception as e:
                    print(f"    ❌ Failed: {e}")
                    results.append((category, url, pdf_path, 0, 0, str(e)))

                finally:
                    await page.close()

        await browser.close()

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("📊 SUMMARY")
    print(f"{'='*60}")
    total_pdfs = 0
    for category, url, pdf_path, pages, chars, err in results:
        status = f"✅ {pages}pp / {chars:,} chars" if not err else f"❌ {err[:60]}"
        print(f"  {pdf_path.name}")
        print(f"    {status}")
        if not err:
            total_pdfs += 1

    print(f"\n✅ {total_pdfs}/{len(results)} PDFs saved to '{PDF_DIR}/' folder")
    return results


await scrape_all()


📁 Category: visa_types  (1 URL)

  [1/1] https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html
    💾 → PDFs/visa_types__1__travel_state_gov_content_travel_en_us-visas_visa-information-resources_all-visa-.pdf
    📂 Accordion headers found: 2
    ✅ 5 pages | 5,565 chars

📁 Category: ds160_application  (4 URLs)

  [1/4] https://ceac.state.gov/genniv/
    💾 → PDFs/ds160_application__1__ceac_state_gov_genniv.pdf
    ✅ 1 pages | 1,932 chars

  [2/4] https://ceac.state.gov/genniv/Common/Instructions.aspx
    💾 → PDFs/ds160_application__2__ceac_state_gov_genniv_Common_Instructions_aspx.pdf
    ⚠️  networkidle timed out (Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://ceac.state.gov/genniv/Common/Instructions.aspx", waiting until "networkidle"
), continuing anyway...
    ✅ 1 pages | 347 chars

  [3/4] https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-ap

[('visa_types',
  'https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html',
  PosixPath('PDFs/visa_types__1__travel_state_gov_content_travel_en_us-visas_visa-information-resources_all-visa-.pdf'),
  5,
  5565,
  None),
 ('ds160_application',
  'https://ceac.state.gov/genniv/',
  PosixPath('PDFs/ds160_application__1__ceac_state_gov_genniv.pdf'),
  1,
  1932,
  None),
 ('ds160_application',
  'https://ceac.state.gov/genniv/Common/Instructions.aspx',
  PosixPath('PDFs/ds160_application__2__ceac_state_gov_genniv_Common_Instructions_aspx.pdf'),
  1,
  347,
  None),
 ('ds160_application',
  'https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application.html',
  PosixPath('PDFs/ds160_application__3__travel_state_gov_content_travel_en_us-visas_visa-information-resources_forms_ds-.pdf'),
  1,
  1653,
  None),
 ('ds160_application',
  'https://travel.state.gov/content/travel/en/us-

US Travel Docs Website -> PDF

In [9]:
PDF_DIR = Path("PDFs")
PDF_DIR.mkdir(exist_ok=True)

URLS = [
    "https://www.ustraveldocs.com/in/en/step-1",
    "https://www.ustraveldocs.com/in/en/step-2",
    "https://www.ustraveldocs.com/in/en/step-3",
    "https://www.ustraveldocs.com/in/en/step-4",
    "https://www.ustraveldocs.com/in/en/step-5",
    "https://www.ustraveldocs.com/in/en/step-6",
]


async def expand_usa_accordions(page):
    """Expand all usa-accordion__button.card-button-content sections."""

    # Count before
    total = await page.evaluate("""
        () => document.querySelectorAll('button.usa-accordion__button.card-button-content').length
    """)
    print(f"    📂 usa-accordion buttons found: {total}")

    if total == 0:
        # Fallback: try just usa-accordion__button in case class combo differs
        total_fallback = await page.evaluate("""
            () => document.querySelectorAll('button.usa-accordion__button').length
        """)
        print(f"    📂 Fallback usa-accordion__button found: {total_fallback}")

    # Click each collapsed button
    clicked = await page.evaluate("""
        async () => {
            // Try specific class first, then broader fallback
            let buttons = document.querySelectorAll(
                'button.usa-accordion__button.card-button-content'
            );
            if (buttons.length === 0) {
                buttons = document.querySelectorAll('button.usa-accordion__button');
            }

            let clicked = 0;
            for (const btn of buttons) {
                const isExpanded = btn.getAttribute('aria-expanded') === 'true';
                if (!isExpanded) {
                    btn.click();
                    clicked++;
                    await new Promise(r => setTimeout(r, 300));
                }
            }
            return clicked;
        }
    """)
    print(f"    ✅ Clicked {clicked} collapsed buttons")

    await asyncio.sleep(2)

    # JS override — force-open any stubbornly collapsed panels
    await page.evaluate("""
        () => {
            // Force all accordion buttons to aria-expanded=true
            document.querySelectorAll('button.usa-accordion__button').forEach(btn => {
                btn.setAttribute('aria-expanded', 'true');
            });

            // Force all accordion content panels to be visible
            document.querySelectorAll('.usa-accordion__content').forEach(el => {
                el.hidden = false;
                el.style.display = 'block';
                el.style.maxHeight = 'none';
                el.style.overflow = 'visible';
            });
        }
    """)

    await asyncio.sleep(1)

    # Verify
    still_collapsed = await page.evaluate("""
        () => [...document.querySelectorAll('button.usa-accordion__button')]
                .filter(b => b.getAttribute('aria-expanded') !== 'true').length
    """)
    print(f"    🔒 Still collapsed after JS override: {still_collapsed}")


async def scrape_ustraveldocs():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        for url in URLS:
            step = url.split("/")[-1]  # e.g. "step-1"
            pdf_path = PDF_DIR / f"visa_process_ustraveldocs__{step}.pdf"

            print(f"\n{'='*55}")
            print(f"🌐 {url}")
            print(f"💾 → {pdf_path}")

            page = await browser.new_page(viewport={"width": 1280, "height": 900})

            try:
                await page.goto(url, wait_until="networkidle", timeout=30000)
            except Exception as e:
                print(f"    ⚠️  networkidle timeout ({e}), continuing...")

            await asyncio.sleep(2)
            await expand_usa_accordions(page)

            await page.pdf(
                path=str(pdf_path),
                format="A4",
                print_background=True,
                margin={"top": "1cm", "bottom": "1cm", "left": "1cm", "right": "1cm"},
            )

            doc = pymupdf.open(str(pdf_path))
            chars = sum(len(pg.get_text()) for pg in doc)
            pages = len(doc)
            doc.close()

            print(f"    ✅ Saved — {pages} pages | {chars:,} chars")
            await page.close()

        await browser.close()

    print(f"\n{'='*55}")
    print(f"✅ All done — PDFs saved to '{PDF_DIR}/' folder")


await scrape_ustraveldocs()


🌐 https://www.ustraveldocs.com/in/en/step-1
💾 → PDFs/visa_process_ustraveldocs__step-1.pdf
    📂 usa-accordion buttons found: 0
    📂 Fallback usa-accordion__button found: 4
    ✅ Clicked 4 collapsed buttons
    🔒 Still collapsed after JS override: 0
    ✅ Saved — 8 pages | 7,172 chars

🌐 https://www.ustraveldocs.com/in/en/step-2
💾 → PDFs/visa_process_ustraveldocs__step-2.pdf
    📂 usa-accordion buttons found: 6
    ✅ Clicked 6 collapsed buttons
    🔒 Still collapsed after JS override: 0
    ✅ Saved — 8 pages | 12,200 chars

🌐 https://www.ustraveldocs.com/in/en/step-3
💾 → PDFs/visa_process_ustraveldocs__step-3.pdf
    📂 usa-accordion buttons found: 3
    ✅ Clicked 3 collapsed buttons
    🔒 Still collapsed after JS override: 0
    ✅ Saved — 8 pages | 8,992 chars

🌐 https://www.ustraveldocs.com/in/en/step-4
💾 → PDFs/visa_process_ustraveldocs__step-4.pdf
    📂 usa-accordion buttons found: 10
    ✅ Clicked 10 collapsed buttons
    🔒 Still collapsed after JS override: 0
    ✅ Saved — 10 pa

Chunking for US Travel Docs

In [ ]:
"""
travel.state.gov RAG scraper 
"""

import asyncio
import json
import re
from pathlib import Path
from playwright.async_api import async_playwright

# ── Config ─────────────────────────────────────────────────────────────────────

URLS = {
    "visa_types": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html",
    ],
    "ds160_application": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application.html",
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application/ds-160-faqs.html",
    ],
    "photo_requirements": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/photos.html",
    ],
    "visa_fees": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/fees/fees-visa-services.html",
    ],
    "administrative_processing": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/administrative-processing-information.html",
    ],
    "visa_denials": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/visa-denials.html",
    ],
    "visa_wait_times": [
        "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/wait-times.html",
    ],
}

OUTPUT_FILE   = Path("chunks_stategov.jsonl")
CHUNK_SIZE    = 800   # target max chars per prose/faq chunk
CHUNK_OVERLAP = 80    # chars of tail from previous chunk seeded into next
MIN_BODY_LEN  = 60    # discard chunks whose body is shorter than this

# ── FIX 1: Expanded nav/footer heading blocklist ──────────────────────────────
# Every heading string that is pure sidebar navigation or footer link lists.
# Lowercase-normalised for comparison.
NAV_HEADINGS = {
    # Top-level site nav injected on every page
    "u.s. visas",
    "english",
    # Sidebar link list
    "visa information & resources",
    "find u.s. embassies & consulates",
    # Footer "More Information" link block
    "more information",
    # "About Wait Times" footer block
    "about wait times",
}

# ── Boilerplate substrings (case-insensitive, body-level filter) ───────────────
BOILERPLATE_PATTERNS = [
    "for consular information or assistance, call the department of state",
    "americans in the middle east",
    "have a question about a u.s. visa",
    "bureau of consular affairs",
    "privacy policy",
    "accessibility statement",
    "skip to main content",
    # Footer links that bleed into prose chunks
    "rights and protections for temporary workers - english",
    "rights and protections for temporary workers - french",
    "global visa wait times\nrights and protections",
    "a-z index\nlatest news",
    "intercountry adoption\ninternational parental child abduction",
    "other visa categories\nu.s. visa: reciprocity and civil documents",
]

# ── Page preparation ───────────────────────────────────────────────────────────

async def clean_page(page):
    """Strip navigation chrome, banners, footers, hidden elements."""
    await page.evaluate("""
    () => {
        [
            "nav", "footer", "header", "aside",
            ".breadcrumb", ".skip-nav", ".skip-link",
            ".tsg-rwd-header", ".tsg-rwd-footer",
            ".tsg-rwd-global-footer", ".tsg-rwd-global-header",
            ".tsg-rwd-utility-nav", ".tsg-rwd-side-nav",
            ".tsg-rwd-breadcrumb", ".share-this", ".social-share",
            "[class*='social']", "[class*='share']",
            "[class*='utility']", "[class*='banner']",
            "[id*='nav']", "[id*='footer']", "[id*='header']", "[id*='sidebar']"
        ].forEach(sel =>
            document.querySelectorAll(sel).forEach(e => e.remove())
        );
        document.querySelectorAll("[aria-hidden='true']").forEach(e => e.remove());
    }
    """)

# ── Accordion expansion ────────────────────────────────────────────────────────

async def expand_accordions(page):
    """
    Two-pass expand:
    1. Click each header so JS fires and content renders.
    2. Force-open anything still collapsed via attribute/style override.
    """
    headers = await page.query_selector_all(".tsg-rwd-accordion-header")
    for h in headers:
        try:
            await h.scroll_into_view_if_needed()
            await h.click()
            await asyncio.sleep(0.15)
        except Exception:
            pass

    await page.evaluate("""
    () => {
        document.querySelectorAll(".tsg-rwd-accordion-header").forEach(el => {
            el.setAttribute("aria-expanded", "true");
            el.classList.add("active");
        });
        document.querySelectorAll(".tsg-rwd-accordion-content").forEach(el => {
            el.style.display   = "block";
            el.style.maxHeight = "none";
            el.style.overflow  = "visible";
            el.hidden          = false;
        });
        document.querySelectorAll("[aria-expanded='false']").forEach(el =>
            el.setAttribute("aria-expanded", "true")
        );
        document.querySelectorAll("details").forEach(el => { el.open = true; });
    }
    """)
    await asyncio.sleep(1.5)
    print(f"    🔓 Accordions expanded: {len(headers)}")

# ── DOM block extraction ───────────────────────────────────────────────────────

async def extract_blocks(page) -> list[dict]:
    """
    Walk DOM in document order and return typed blocks:

      heading      – h1-h4 or accordion header NOT ending in "?"
      faq_question – heading / accordion header ending in "?"      (FIX 2)
      table_raw    – list of row strings, one per <tr>             (FIX 4)
      text         – p, li, dt, dd  body content
    """
    blocks = await page.evaluate("""
    () => {
        // FIX 4: convert a <table> into one string per data row
        function tableToRows(table) {
            const rows = Array.from(table.querySelectorAll("tr"));
            if (!rows.length) return [];
            const headers = Array.from(rows[0].querySelectorAll("th,td"))
                .map(x => x.innerText.trim());
            return rows.slice(1).map(r => {
                const cells = Array.from(r.querySelectorAll("td"));
                return cells
                    .map((c, i) => (headers[i] || "col" + i) + ": " + c.innerText.trim())
                    .filter(p => p.replace(/[:,\\s]/g, "").length > 0)
                    .join(" | ");
            }).filter(row => row.length > 5);
        }

        const results = [];
        const seen    = new Set();

        const nodes = document.querySelectorAll(
            ".tsg-rwd-accordion-header, h1, h2, h3, h4, table, p, li, dt, dd"
        );

        nodes.forEach(el => {
            const tag      = el.tagName.toLowerCase();
            const isAccHdr = el.classList.contains("tsg-rwd-accordion-header");
            const isHTag   = ["h1","h2","h3","h4"].includes(tag);

            // Skip accordion header children and table cells
            if (!isAccHdr && el.closest(".tsg-rwd-accordion-header")) return;
            if (tag !== "table" && el.closest("table")) return;
            if (tag === "li" && el.closest("p")) return;

            const raw = el.innerText.trim();
            if (raw.length < ((isAccHdr || isHTag) ? 3 : 20)) return;

            const fp = raw.slice(0, 120);
            if (seen.has(fp)) return;
            seen.add(fp);

            if (isAccHdr || isHTag) {
                // FIX 2: headings ending with "?" become faq_question
                const isFaq = raw.trim().endsWith("?");
                results.push({ type: isFaq ? "faq_question" : "heading", text: raw });
            } else if (tag === "table") {
                const rows = tableToRows(el);
                if (rows.length) results.push({ type: "table_raw", rows });
            } else {
                results.push({ type: "text", text: raw });
            }
        });

        return results;
    }
    """)
    return blocks

# ── Text helpers ───────────────────────────────────────────────────────────────

def dedup_lines(text: str) -> str:
    """Remove duplicate lines caused by mobile/desktop DOM copies."""
    seen, out = set(), []
    for line in text.split("\n"):
        key = line.strip()
        if key and key not in seen:
            seen.add(key)
            out.append(line)
    return "\n".join(out)


def clean_text(text: str) -> str:
    text = re.sub(r'\bNaN\b', '', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ── FIX 3: Rolling-buffer sentence splitter ────────────────────────────────────
#
# The root cause of mid-word fragments was: Playwright's innerText already
# returns text that was soft-wrapped or truncated at a DOM boundary, so the
# very first character of a "text" block might be mid-word (e.g. "es where…").
# The old code split each block independently — if a block started mid-word,
# every split piece from it also started mid-word.
#
# The fix: accumulate ALL text blocks for a section into one raw_body string
# first, THEN split that string at sentence boundaries.  The sentence-boundary
# regex only breaks at ". ", "? ", "! ", "\n\n" — never mid-word.
# Each resulting piece re-gets the heading prepended so it is self-contained.

def split_at_sentence_boundaries(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Split `text` into pieces ≤ chunk_size, breaking only at sentence ends.

    Two critical correctness properties:
    1. Split pattern requires the next token to start with uppercase/digit,
       preventing splits inside abbreviations like "U.S.", "Dr.", "i.e." etc.
    2. Overlap is seeded from the last complete sentence(s) of the previous
       chunk — never a raw character-slice — so no chunk ever starts mid-word
       or mid-sentence.
    """
    # Only split where sentence-ending punctuation is followed by whitespace
    # then an uppercase letter, digit, or opening quote/bracket.
    # This avoids splitting "U.S. law" at "U." or "S."
    sentences = re.split(r'(?<=[.?!])\s+(?=[A-Z0-9\"\'\(\[])', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks: list[str] = []
    current_sents: list[str] = []

    for sent in sentences:
        candidate = " ".join(current_sents + [sent])
        if len(candidate) <= chunk_size:
            current_sents.append(sent)
        else:
            if current_sents:
                chunks.append(" ".join(current_sents))
            # Seed overlap from the tail of current_sents using whole sentences,
            # not a character slice — guarantees the next chunk starts at a
            # sentence boundary.
            seed: list[str] = []
            budget = overlap * 3  # generous window (up to ~240 chars)
            for s in reversed(current_sents):
                if len(" ".join([s] + seed)) <= budget:
                    seed.insert(0, s)
                else:
                    break
            current_sents = seed + [sent]

    if current_sents:
        chunks.append(" ".join(current_sents))

    return chunks

# ── Chunk builder ──────────────────────────────────────────────────────────────

def build_chunks(blocks: list[dict], url: str, category: str) -> list[dict]:
    """
    Converts DOM blocks into logical chunks using a rolling body buffer.

    heading      → flush buffer as prose, start new section
    faq_question → flush buffer, then accumulate body as FAQ answer  (FIX 2)
    table_raw    → flush buffer, emit one chunk per row               (FIX 4)
    text         → append to rolling body buffer (never split here)

    Splitting happens only at flush time, on the complete accumulated body.
    This prevents mid-word artifacts from DOM-level text boundaries.        (FIX 3)
    """
    chunks     = []
    heading    = ""
    body_parts : list[str] = []
    in_faq     = False
    faq_q      = ""

    def make_chunk(h: str, body: str, ctype: str,
                   split_part: int = 0, split_total: int = 0) -> dict | None:
        body = dedup_lines(clean_text(body))
        if len(body) < MIN_BODY_LEN:
            return None
        text = (h + "\n" + body).strip() if h else body
        c = {
            "heading":      h,
            "text":         text,
            "content_type": ctype,
            "source_url":   url,
            "domain":       "travel.state.gov",
            "category":     category,
        }
        if split_total > 1:
            c["split_part"]  = split_part
            c["split_total"] = split_total
        return c

    def flush():
        """
        Emit the accumulated body as one or more chunks.
        Splitting is done here on the full body string — never on individual
        DOM text blocks — so sentence-boundary splits are always clean.
        """
        nonlocal body_parts, in_faq, faq_q
        raw_body = "\n".join(body_parts).strip()
        body_parts = []

        h     = faq_q if in_faq else heading
        ctype = "faq"  if in_faq else "prose"

        if not raw_body:
            if in_faq:
                in_faq = False
                faq_q  = ""
            return

        if len(raw_body) <= CHUNK_SIZE:
            c = make_chunk(h, raw_body, ctype)
            if c:
                chunks.append(c)
        else:
            pieces = split_at_sentence_boundaries(raw_body, CHUNK_SIZE, CHUNK_OVERLAP)
            valid  = [p.strip() for p in pieces if len(p.strip()) >= MIN_BODY_LEN]
            for i, piece in enumerate(valid):
                c = make_chunk(h, piece, ctype,
                               split_part=i + 1, split_total=len(valid))
                if c:
                    chunks.append(c)

        if in_faq:
            in_faq = False
            faq_q  = ""

    for block in blocks:
        btype = block["type"]

        if btype == "heading":
            flush()
            heading = block["text"]

        elif btype == "faq_question":
            # FIX 2: glue question to its answer body
            flush()
            in_faq = True
            faq_q  = block["text"]

        elif btype == "table_raw":
            # FIX 4: emit one chunk per table row
            flush()
            ctx = faq_q if in_faq else heading
            for row in block["rows"]:
                row = row.strip()
                if len(row) < MIN_BODY_LEN:
                    continue
                chunks.append({
                    "heading":      ctx,
                    "text":         (ctx + "\n" + row).strip() if ctx else row,
                    "content_type": "table_row",
                    "source_url":   url,
                    "domain":       "travel.state.gov",
                    "category":     category,
                })

        else:  # text node — just accumulate
            body_parts.append(block["text"])

    flush()  # flush final section
    return chunks

# ── Filtering ──────────────────────────────────────────────────────────────────

def is_nav(chunk: dict) -> bool:
    """FIX 1: exact heading match — no heuristics."""
    return chunk["heading"].strip().lower() in NAV_HEADINGS


def is_boilerplate(chunk: dict) -> bool:
    lower = chunk["text"].lower()
    return any(p in lower for p in BOILERPLATE_PATTERNS)


def deduplicate(chunks: list[dict]) -> list[dict]:
    """Fingerprint on first 200 chars of body text (heading-agnostic)."""
    seen, out = set(), []
    for c in chunks:
        body = c["text"].replace(c["heading"], "", 1).strip()
        fp   = body[:200]
        if fp not in seen:
            seen.add(fp)
            out.append(c)
    return out

# ── Per-URL pipeline ───────────────────────────────────────────────────────────

async def scrape_url(page, url: str, category: str) -> list[dict]:
    print(f"\n  🌐 {url}")
    try:
        await page.goto(url, wait_until="networkidle", timeout=30_000)
    except Exception as e:
        print(f"    ⚠️  networkidle timeout ({e}), continuing...")

    await asyncio.sleep(2)
    await clean_page(page)
    await expand_accordions(page)

    blocks = await extract_blocks(page)
    print(f"    📦 Raw blocks  : {len(blocks)}")

    chunks = build_chunks(blocks, url, category)

    before = len(chunks)
    chunks = [c for c in chunks if not is_nav(c)]
    chunks = [c for c in chunks if not is_boilerplate(c)]
    print(f"    🗑️  Noise removed: {before - len(chunks)}")

    chunks = deduplicate(chunks)
    print(f"    ✅ Final chunks : {len(chunks)}")
    return chunks

# ── Main ───────────────────────────────────────────────────────────────────────

async def scrape_all():
    all_chunks: list[dict] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        for category, urls in URLS.items():
            print(f"\n{'='*60}")
            print(f"📁 {category}")
            for url in urls:
                page = await browser.new_page(viewport={"width": 1280, "height": 900})
                try:
                    chunks = await scrape_url(page, url, category)
                    all_chunks.extend(chunks)
                except Exception as e:
                    print(f"    ❌ Failed: {e}")
                finally:
                    await page.close()

        await browser.close()

    # ── Save ──────────────────────────────────────────────────────────────────
    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for c in all_chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")

    # ── Summary ───────────────────────────────────────────────────────────────
    from collections import Counter
    print(f"\n{'='*60}")
    print(f"✅ Total chunks : {len(all_chunks)}")
    print(f"✅ Saved to     : {OUTPUT_FILE}\n")

    print(f"{'─'*60}")
    print(f"  {'category':<35} {'chunks':>6}  content_types")
    print(f"{'─'*60}")
    for cat in URLS:
        cc     = [c for c in all_chunks if c["category"] == cat]
        ctypes = Counter(c.get("content_type", "?") for c in cc)
        bd     = "  ".join(f"{t}:{n}" for t, n in sorted(ctypes.items()))
        print(f"  {cat:<35} {len(cc):>6}  {bd}")

    # ── Spot-check: flag any remaining mid-sentence starts ────────────────────
    print(f"\n{'─'*60}")
    print("🔍 Fragment spot-check (body starts with lowercase after heading):")
    flagged = 0
    for c in all_chunks:
        if c.get("split_part", 1) < 2:
            continue
        body = c["text"].replace(c["heading"], "", 1).strip()
        first = body.split()[0] if body.split() else ""
        if first and first[0].islower():
            print(f"   [{c['content_type']}] {c['heading'][:40]} → {body[:60]!r}")
            flagged += 1
    print(f"   {'None ✅' if not flagged else f'{flagged} flagged ⚠️'}")

    # ── Spot-check: short bodies ──────────────────────────────────────────────
    print(f"\n🔍 Short-body spot-check (<{MIN_BODY_LEN} chars):")
    flagged2 = 0
    for c in all_chunks:
        body = c["text"].replace(c["heading"], "", 1).strip()
        if len(body) < MIN_BODY_LEN:
            print(f"   [{c['content_type']}] {c['heading'][:40]} → {body!r}")
            flagged2 += 1
    print(f"   {'None ✅' if not flagged2 else f'{flagged2} flagged ⚠️'}")

    return all_chunks


# ── Entry point (works in Jupyter and as script) ───────────────────────────────
all_chunks = await scrape_all()


📁 visa_types

  🌐 https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html
    🔓 Accordions expanded: 2
    📦 Raw blocks  : 131
    🗑️  Noise removed: 11
    ✅ Final chunks : 55

📁 ds160_application

  🌐 https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application.html
    🔓 Accordions expanded: 0
    📦 Raw blocks  : 122
    🗑️  Noise removed: 13
    ✅ Final chunks : 0

  🌐 https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/forms/ds-160-online-nonimmigrant-visa-application/ds-160-faqs.html
    🔓 Accordions expanded: 25
    📦 Raw blocks  : 213
    🗑️  Noise removed: 10
    ✅ Final chunks : 35

📁 photo_requirements

  🌐 https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/photos.html
    🔓 Accordions expanded: 0
    📦 Raw blocks  : 176
    🗑️  Noise removed: 10
    ✅ Final chunks : 14

📁 visa_fees

  🌐 https://travel.sta